## LABORATORY EXERCISE 10

# CLOUD COMPUTING AND AI MODEL DEPLOYMENT FOR ENGINEERING APPLICATIONS

# CE18 – CONCRETE BRIDGE DECK CRACK DETECTION
## Classification: CRACKED VS NON-CRACKED

# PART I — DATASET PREPARATION

## We separated the dataset to prevent the model from being evaluated using the same images it learned from. Training data is used for learning, validation data is used for monitoring and improving the model during training, while test data is reserved for the final unbiased evaluation. We used 70%  to train, 15% to test and 15% to validate.

## STEP 1: Import Required Libraries

In [46]:
import os
import shutil
import random
from pathlib import Path

# STEP 2: Define Dataset Paths

In [49]:
SOURCE_DIR = Path("dataset/D")

CD_DIR = SOURCE_DIR / "CD"
UD_DIR = SOURCE_DIR / "UD"

TRAIN_DIR = Path("dataset/train")
VAL_DIR = Path("dataset/valid")
TEST_DIR = Path("dataset/test")

print("Source dataset:", SOURCE_DIR)
print("Cracked Deck:", CD_DIR)
print("Uncracked Deck:", UD_DIR)

Source dataset: dataset\D
Cracked Deck: dataset\D\CD
Uncracked Deck: dataset\D\UD


## STEP 3: Check the Original Dataset

In [50]:
print("CD exists:", CD_DIR.exists())
print("UD exists:", UD_DIR.exists())

cd_images = list(CD_DIR.glob("*"))
ud_images = list(UD_DIR.glob("*"))

print("Cracked Deck images:", len(cd_images))
print("Uncracked Deck images:", len(ud_images))

CD exists: True
UD exists: True
Cracked Deck images: 2025
Uncracked Deck images: 11595


# STEP 4: Create Training, Validation and Testing Folders

In [53]:
for folder in [
    TRAIN_DIR / "CD",
    TRAIN_DIR / "UD",
    VAL_DIR / "CD",
    VAL_DIR / "UD",
    TEST_DIR / "CD",
    TEST_DIR / "UD"
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Train, validation and test folders created.")

Train, validation and test folders created.


# STEP 5: Split the Dataset

In [54]:
def split_and_copy(source_folder, class_name, train_ratio=0.70,
                   val_ratio=0.15, test_ratio=0.15):

    images = [
        f for f in source_folder.iterdir()
        if f.is_file()
    ]

    random.seed(42)
    random.shuffle(images)

    total = len(images)

    train_end = int(total * train_ratio)
    val_end = train_end + int(total * val_ratio)

    train_images = images[:train_end]
    val_images = images[train_end:val_end]
    test_images = images[val_end:]

    print(f"\n{class_name}")
    print("Total:", total)
    print("Training:", len(train_images))
    print("Validation:", len(val_images))
    print("Testing:", len(test_images))

    for image in train_images:
        shutil.copy2(
            image,
            TRAIN_DIR / class_name / image.name
        )

    for image in val_images:
        shutil.copy2(
            image,
            VAL_DIR / class_name / image.name
        )

    for image in test_images:
        shutil.copy2(
            image,
            TEST_DIR / class_name / image.name
        )


split_and_copy(CD_DIR, "CD")
split_and_copy(UD_DIR, "UD")


CD
Total: 2025
Training: 1417
Validation: 303
Testing: 305

UD
Total: 11595
Training: 8116
Validation: 1739
Testing: 1740


# STEP 6: Verify the Dataset Split

In [55]:
def count_images(folder):
    return len([
        f for f in folder.iterdir()
        if f.is_file()
    ])


print("\n===== DATASET SUMMARY =====")

print("\nTRAIN")
print("CD:", count_images(TRAIN_DIR / "CD"))
print("UD:", count_images(TRAIN_DIR / "UD"))

print("\nVALIDATION")
print("CD:", count_images(VAL_DIR / "CD"))
print("UD:", count_images(VAL_DIR / "UD"))

print("\nTEST")
print("CD:", count_images(TEST_DIR / "CD"))
print("UD:", count_images(TEST_DIR / "UD"))


===== DATASET SUMMARY =====

TRAIN
CD: 1417
UD: 8116

VALIDATION
CD: 303
UD: 1739

TEST
CD: 305
UD: 1740


# PART II — MODEL DEVELOPMENT AND TRAINING

# STEP 1: Import the Libraries

In [56]:
import os
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

# STEP 2: Dataset Paths

In [57]:
train_path = "dataset/train"
val_path = "dataset/val"
test_path = "dataset/test"

# STEP 3: Data Augmentation

In [58]:
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1
)

val_gen = ImageDataGenerator(
    rescale=1./255
)

test_gen = ImageDataGenerator(
    rescale=1./255
)

# STEP 4: Load Dataset

In [60]:
train_data = train_gen.flow_from_directory(
    train_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode="binary",
    shuffle=True
)

val_data = val_gen.flow_from_directory(
    val_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode="binary",
    shuffle=False
)

test_data = test_gen.flow_from_directory(
    test_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode="binary",
    shuffle=False
)

Found 9533 images belonging to 2 classes.
Found 2042 images belonging to 2 classes.
Found 2045 images belonging to 2 classes.


# STEP 5: Verify Dataset

In [61]:
print("Class indices:", train_data.class_indices)

print("Training images:", train_data.samples)
print("Validation images:", valid_data.samples)
print("Testing images:", test_data.samples)

Class indices: {'CD': 0, 'UD': 1}
Training images: 9533
Validation images: 2042
Testing images: 2045


# STEP 6: Calculate Class Weights

## The dataset has many more UD images than CD, so class weighting helps prevent the model from simply favouring the larger class.

In [62]:
classes = np.unique(train_data.classes)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_data.classes
)

class_weight_dict = dict(
    zip(classes, class_weights)
)

print("Class weights:", class_weight_dict)

Class weights: {np.int32(0): np.float64(3.3637967537050106), np.int32(1): np.float64(0.5872966978807295)}


# STEP 7: Build MobileNetV2

In [63]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

## STEP 8: Compile the Model

In [64]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# STEP 9: Callbacks

In [65]:
os.makedirs("models", exist_ok=True)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

model_check = ModelCheckpoint(
    "models/concrete_bridge_crack_classifier.keras",
    monitor="val_loss",
    save_best_only=True
)

# STEP 10: Train the Model

In [67]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    class_weight=class_weight_dict,
    callbacks=[
        early_stop,
        model_check
    ]
)

Epoch 1/20
298/298 ━━━━━━━━━━━━━━━━━━━━ 489s 2s/step - accuracy: 0.7819 - loss: 0.5270 - val_accuracy: 0.8834 - val_loss: 0.3783
Epoch 2/20
298/298 ━━━━━━━━━━━━━━━━━━━━ 462s 2s/step - accuracy: 0.7816 - loss: 0.5228 - val_accuracy: 0.8986 - val_loss: 0.3480
Epoch 3/20
298/298 ━━━━━━━━━━━━━━━━━━━━ 465s 2s/step - accuracy: 0.7991 - loss: 0.5173 - val_accuracy: 0.9001 - val_loss: 0.3530
Epoch 4/20
298/298 ━━━━━━━━━━━━━━━━━━━━ 464s 2s/step - accuracy: 0.7950 - loss: 0.5111 - val_accuracy: 0.8746 - val_loss: 0.4012
Epoch 5/20
298/298 ━━━━━━━━━━━━━━━━━━━━ 522s 2s/step - accuracy: 0.8149 - loss: 0.5027 - val_accuracy: 0.8893 - val_loss: 0.3336
Epoch 6/20
298/298 ━━━━━━━━━━━━━━━━━━━━ 911s 3s/step - accuracy: 0.8054 - loss: 0.5048 - val_accuracy: 0.8546 - val_loss: 0.3994
Epoch 7/20
298/298 ━━━━━━━━━━━━━━━━━━━━ 1070s 4s/step - accuracy: 0.8057 - loss: 0.5041 - val_accuracy: 0.9011 - val_loss: 0.3164
Epoch 8/20
298/298 ━━━━━━━━━━━━━━━━━━━━ 542s 2s/step - accuracy: 0.8036 - loss: 0.5041 - val_acc

# STEP 11: Plot Training Results

In [68]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)

plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")

plt.title("Training and Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()


plt.subplot(1, 2, 2)

plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")

plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.show()

C:\Users\BIG PRESH\AppData\Local\Temp\ipykernel_8312\1786960475.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# STEP 12: Save the Model

In [69]:
MODEL_PATH = "models/concrete_bridge_crack_classifier.keras"

model.save(MODEL_PATH)

print("Model saved successfully:")
print(MODEL_PATH)

Model saved successfully:
models/concrete_bridge_crack_classifier.keras


# STEP 13: Evaluate the Model

In [70]:
test_loss, test_accuracy = model.evaluate(test_data)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

64/64 ━━━━━━━━━━━━━━━━━━━━ 142s 2s/step - accuracy: 0.9120 - loss: 0.2959
Test Loss: 0.2958773076534271
Test Accuracy: 0.9119804501533508


# STEP 14: Classification Report

In [71]:
test_data.reset()

predictions = model.predict(test_data)

predicted_classes = (predictions >= 0.5).astype(int).flatten()

true_classes = test_data.classes

print(
    classification_report(
        true_classes,
        predicted_classes,
        target_names=["Cracked", "Non-Cracked"]
    )
)

64/64 ━━━━━━━━━━━━━━━━━━━━ 85s 1s/step
              precision    recall  f1-score   support

     Cracked       0.82      0.52      0.64       305
 Non-Cracked       0.92      0.98      0.95      1740

    accuracy                           0.91      2045
   macro avg       0.87      0.75      0.79      2045
weighted avg       0.91      0.91      0.90      2045



# STEP 15: Confusion Matrix

In [72]:
cm = confusion_matrix(
    true_classes,
    predicted_classes
)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[ 159  146]
 [  34 1706]]
